# Toba Health Insight — Notebook Baseline M7/M8
**DATA SINTETIS; BUKAN PENELITIAN KLINIS.**
Target `future_7d_mean_sbp`: rata-rata sistolik harian tujuh hari kalender sesudah tanggal t.
Semua fitur memakai informasi t atau sebelumnya. Dua sesi digabung per hari. Split dilakukan berdasarkan ID responden.


In [ ]:
from pathlib import Path
import sys
root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(root / "backend"))
from app.baseline import run_experiment, make_training
from app.pipeline import prepare
import pandas as pd
prof = root / "data/sample/Data_Responden_Sintetis.csv"
days = root / "data/sample/Data_Harian_Sintetis.csv"
print("Sumber:", prof, days)


## 1. Audit data & grain
Profil berisi **19 responden**, catatan harian **570 pasangan ID-tanggal**, dan dua sesi pengukuran menghasilkan **1.140 fakta sesi**. Nama di sumber tidak dimuat ke tabel analitik.


In [ ]:
p, d, s, quality = prepare(pd.read_csv(prof, dtype=str, keep_default_na=False), pd.read_csv(days, dtype=str, keep_default_na=False), synthetic=True)
print("Profil / harian / sesi:", len(p), len(d), len(s))
print("Kualitas:", quality)
assert len(p)==19 and len(d)==570 and len(s)==1140


## 2. Feature engineering & target
Untuk hari t: fitur histori t−6 sampai t, target rata-rata t+1 sampai t+7. Jendela harus 7 hari kalender berturut-turut dan tidak mengandung pembacaan sistolik hilang. Target masih **usulan** dan membutuhkan persetujuan dosen.


In [ ]:
training = make_training(p,d,s)
print("Baris training keseluruhan:", len(training), "ID unik:", training.respondent_id.nunique())
display(training.head())


## 3. Baseline naif dan Linear Regression pembanding
Split berdasarkan *respondent_id*, sehingga tidak satu pun ID hadir pada kedua subset. MAE/RMSE dalam mmHg. R² perlu ditafsirkan bersama ukuran sampel.


In [ ]:
result=run_experiment(str(prof),str(days),str(root / 'models'))
print("Train:",result['rows_training'],"Test:",result['rows_test'])
print("Tidak ada ID bocor:", not (set(result['respondents_train']) & set(result['respondents_test'])))
display(pd.DataFrame(result['scores']))


## 4. Visualisasi galat prediksi
Grafik ini digunakan untuk memeriksa pola residual; bukan diagnosis atau performa pada manusia nyata.


In [ ]:
import matplotlib.pyplot as plt
s=pd.DataFrame(result['actual_vs_predicted'])
fig,ax=plt.subplots(figsize=(7,4))
ax.scatter(s['actual'],s['naive'],alpha=.5,label='Baseline naif')
ax.scatter(s['actual'],s['linear'],alpha=.5,label='Linear Regression')
lo=min(s['actual'].min(),s['naive'].min(),s['linear'].min()); hi=max(s['actual'].max(),s['naive'].max(),s['linear'].max())
ax.plot([lo,hi],[lo,hi],'--',label='Ideal')
ax.set(xlabel='Aktual (mmHg)',ylabel='Prediksi (mmHg)',title='SINTETIS: actual vs predicted')
ax.legend();plt.show()


## 5. Batasan & kelanjutan
M6–M8 mengharuskan dashboard v1, baseline, pembanding awal, metrik dan rencana lanjut. Tahap akhir masih membutuhkan validasi data primer, persetujuan target, tuning, minimal tiga algoritma, analisis error, model card final dan deployment publik. *Jangan campurkan data sintetis dan aktual untuk klaim hasil penelitian.*
